In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [5]:
loader = PyPDFLoader("../data/FullstackResume.pdf")
docs = loader.load()


In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

4

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)


d:\CODING\GENAI_Series\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1311.34it/s]


#### agent = tools, llm, prompt


In [45]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retreive the relevant data of the PDF Documents, and these pdf documents have detailed about a person academic qualification
    """
    docs = vector_store.similarity_search(query=query, k=4)
    context = ""
    print("tool Called", query)
    for doc in docs:
        context = doc.page_content + "\n\n"
    return context

In [42]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


In [38]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    Always use the 'retriever_tool' tool for questions requiring external knowledge.
"""

In [46]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [48]:
query = "what is the name of the company and what are the achievments of the students"
response = agent.invoke({"messages":[{"role":"user","content": query}]})
result = response["messages"][-1].text
print(result)

tool Called company name student achievements
tool Called ACHIEVEMENTS company name
tool Called name education experience company
Based on the provided documents, there is no mention of a specific company name. However, regarding the student's achievements, the document highlights:

* **Competitive Programming and Problem Solving** on **LeetCode**.
